In [ ]:
import pytrinamic
from pytrinamic.connections import ConnectionManager
#from pytrinamic.connections import ConnectionInterface
from pytrinamic.modules import TMCM6110
import time
import scipy.io
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import trange
from itertools import chain

# Motor Control 

In [ ]:
class XYZ():
    def __init__(self):
        self.connectionManager = ConnectionManager()
        self.interface = self.connectionManager.connect()

        # Create an instance of the TMCM_6110 class
        self.module = TMCM6110(self.interface)


        self.motor_0 =  self.module.motors[0]
        self.motor_1 =  self.module.motors[1]
        self.motor_2 =  self.module.motors[2]
        print("Preparing parameters")
        
    def XYZ_setup(self,max_current=500,standby_current=200,boost_current=0,velocity=1500,acceleration=1000,position=0):
        motors_list = []
        for i in [0,1,2]:
            motor_name = f"motor_{i}"
            motor = getattr(self, motor_name)
            # Now you can use 'motor' as a reference to self.motor_0, self.motor_1, etc.
            motor.drive_settings.max_current=max_current
            motor.drive_settings.standby_current=standby_current
            motor.drive_settings.boost_current=boost_current
            motor.drive_settings.microstep_resolution = motor.ENUM.microstep_resolution_256_microsteps
            motor.max_acceleration=acceleration
            motor.max_velocity=velocity
#             motor.actual_position=position
            print(motor)
            motors_list.append(motor)
        return motors_list[2],motors_list[1],motors_list[0],self.interface
        
  

# Capturing functions


In [ ]:
def capture_location(scope, pt_exp, exp_name, x, y, num=1000):
    """
    Captures and stores traces for a given (x, y) location.

    Parameters:
    scope: Object responsible for capturing traces.
    pt_exp: Experiment dataset handler for plaintexts and keys.
    exp_name: Dataset handler where captured traces will be stored.
    x, y: Coordinates representing the capture location.
    num (int, optional): Number of traces to capture (default is 1000).
    """

    # Retrieve key, random plaintext, and fixed plaintext datasets
    keys_pt = pt_exp.get_dataset("keys").read_data(0, num)
    random_pt = pt_exp.get_dataset("plaintexts").read_data(0, num)
    fixed_pt = pt_exp.get_dataset("fixed_pt").read_data(0, num)

    # Capture traces using test vector leakage assessment (TVLA) method
    f, r = scope.capture_traces_tvla(num, keys_pt, fixed_pt, keys_pt, random_pt)

    # Store captured traces for the given location
    print("Storing for location: " + str(x) + "_" + str(y))
    exp_name.add_dataset("fixed_" + str(x) + "_" + str(y), f, datatype="float32")
    exp_name.add_dataset("random_" + str(x) + "_" + str(y), r, datatype="float32")

    print("Traces stored")

    return None


def Grid_Tracing_scapegoat(X_range,Y_range,X_number_of_step,Y_number_of_step,X,Y,Z,interface,scope,pt_exp,exp_store,number_of_traces):
    cordinate_traces={}
    X_moment=0
    Y_moment=0
    capture_location(scope,pt_exp,exp_store,X_moment,Y_moment,number_of_traces)
    X_start_position=X.get_actual_position()
    Y_start_position=Y.get_actual_position()
    print(f"Starting Postion - ({X_start_position},{Y_start_position})")
    while Y_moment<=Y_number_of_step:
        X_initial_position=X.get_actual_position()
        Y_initial_position=Y.get_actual_position()
        for _ in range(X_number_of_step):
            X.move_by(X_range)
            print(f'Moving X to {X_initial_position + X_range}')
            while X.get_actual_position() != X_initial_position + X_range:
                time.sleep(0.1)
            X_moment += 1
            capture_location(scope,pt_exp,exp_store,X_moment,Y_moment,number_of_traces)
            X_initial_position=X.get_actual_position()
        if Y_moment==Y_number_of_step:
            break
        Y.move_by(Y_range)
        Y_moment+=1
        print(f'Moving Y to {Y_initial_position + Y_range}')
        while Y.get_actual_position() != Y_initial_position + Y_range:
            time.sleep(0.1)
        capture_location(scope,pt_exp,exp_store,X_moment,Y_moment,number_of_traces)
        Y_initial_position=Y.get_actual_position()
        for _ in range(X_number_of_step):
            X.move_by(-X_range)
            print(f'Moving X to {X_initial_position - X_range}')
            while X.get_actual_position() != X_initial_position - X_range:
                time.sleep(0.1)
            X_moment -= 1
            capture_location(scope,pt_exp,exp_store,X_moment,Y_moment,number_of_traces)
            X_initial_position=X.get_actual_position()
        if Y_moment==Y_number_of_step:
            break
        # Move down 1 step
        Y.move_by(Y_range)
        Y_moment += 1
        print(f'Moving Y to {Y_initial_position + Y_range}')
        while Y.get_actual_position() != Y_initial_position + Y_range:
            time.sleep(0.1)
        capture_location(scope,pt_exp,exp_store,X_moment,Y_moment,number_of_traces)
        Y_initial_position=Y.get_actual_position()
    X.move_to(X_start_position)
    while X.get_actual_position() != X_start_position:
        time.sleep(0.1)
    Y.move_to(Y_start_position)
    while Y.get_actual_position() != Y_start_position:
        time.sleep(0.1)
    print(f"Final Postion - ({X.get_actual_position()},{Y.get_actual_position()})")
    return None

# Metrics (one-click)

In [ ]:
def plot_CEMA_heatmap(test,pt_exp,num,target_byte =0, grid_size=5):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    keys = pt_exp.get_dataset("keys").read_data(0,num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0,num)
    # Initialize the t_values array directly
    CEMA_values = np.zeros((grid_size, grid_size))
    CEMA_guesses = np.zeros((grid_size, grid_size))

    # Calculate t-statistics and populate the t_values array in a single loop
    for i in range(grid_size):
        for j in trange(grid_size):
            print(f"loop_{i}_{j}")

            traces = test.get_dataset( f"random_{i}_{j}").read_data(0,num)
            t, t_max = scapegoat_cpa_byte(traces,keys,plaintexts,target_byte)
            CEMA_values[i, j] = t_max  # Store the maximum absolute t-statistic
            CEMA_guesses[i,j] = t

    # Rotate the heatmap by transposing the CEMA_values array
    CEMA_values_rotated = np.rot90(CEMA_values, k=3)  # Rotate by 90 degrees clockwise (k=3)
    CEMA_guesses_rotated = np.rot90(CEMA_guesses, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title
    plt.title("Heatmap of CEMA")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()
    return CEMA_guesses_rotated,CEMA_values_rotated

def plot_CEMA_wr(test,pt_exp,num,target_byte =0, x=0,y =0,div=10):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    keys = pt_exp.get_dataset("keys").read_data(0,num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0,num)
    # Initialize the t_values array directly
    traces = test.get_dataset( f"random_{x}_{y}").read_data(0,num)


    maxcpa_matrix = np.zeros((int(num/div), 256))  # Store all CPA values
    iterations = 1
    for i in trange(1,num):
        if i%div==0:
            for k in range(256):
                leakage = leakage_model_hamming_weight(num_traces=i, plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
                correlation = pearson_correlation(leakage, traces[:i])
                maxcpa_matrix[int(i/div), k] = (np.nanmax(np.abs(correlation)))  # Store CPA value
            iterations= iterations + 1

    # Debugging: Check matrix shape
    print("Shape of maxcpa_matrix:", maxcpa_matrix.shape)

    # Adjust x to match available samples
    xp = np.arange(2, min(iterations + 2, maxcpa_matrix.shape[0]))  # Avoid going out of bounds
    plt.figure(figsize=(10, 6))
    # Ensure 'maxcpa_matrix' has valid dimensions
    if maxcpa_matrix.shape[0] > 2:  # Relaxed condition
        # Plot threshold line
        plt.plot(xp, (abs(4)/np.sqrt(xp * div)) * np.ones_like(xp), 
                 color="black", linestyle='dotted', linewidth=1.5, label="Threshold")

        # Plot all 256 traces, starting from sample 2
        for i in range(256):
            if i == 43:
                plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="red", 
                         alpha=0.9, linewidth=1.5, label="Correct key" if i == 43 else "")
            else:
                plt.plot(xp, maxcpa_matrix[2:len(xp) + 2, i], color="grey", 
                         alpha=0.1, linewidth=0.5, label="Wrong keys" if i == 0 else "")


    plt.xlabel("No. of traces x"+ str(div),fontsize =14)
    plt.ylabel("Max CPA Value",fontsize =14)
    plt.title("Correlation Power Analysis : ("+str(x)+","+str(y)+")",fontsize =18)
    plt.yticks(fontsize = 12)
    plt.xticks(fontsize = 12)
    plt.legend()
    plt.show()
    return maxcpa_matrix

def plot_SNR_heatmap(test,pt_exp,num,target_byte =0, grid_size=5,SNR_type = "BYTE"):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    keys = pt_exp.get_dataset("keys").read_data(0,num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0,num)
    if SNR_type == "BYTE" :
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL"    :
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1] 
    else :
        print("incorrect SNR type")
        return -1

    CEMA_values = np.zeros((grid_size, grid_size))

    labelsUnique = np.unique(labels)

    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {}
            for k in labelsUnique:
                sorted_labels[k] = []
            print(f"loop_{i}_{j}")

            traces = test.get_dataset( f"random_{i}_{j}").read_data(0,num)
            
            # organize labels using traces, keys and plaintexts
            # add traces to labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # run metric and visualize result. Save it in the visualization folder. 
            t = signal_to_noise_ratio(sorted_labels )#, visualize=True, visualization_path=test.get_visualization_path() + f"SNR_{i}_{j}")
            
            CEMA_values[i, j] =  np.nanmax(np.abs(t))   # Store the maximum absolute t-statistic

    # Rotate the heatmap by transposing the CEMA_values array
    CEMA_values_rotated = np.rot90(CEMA_values, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title
    plt.title("Heatmap of SNR all bytes")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()
    return CEMA_values_rotated  ,  10 * np.log10(CEMA_values_rotated)


def generate_box_plots(test, pt_exp, num_list, target_byte=0, grid_size=5):
    """
    Generate multiple box plots for different numbers of traces.

    Parameters:
    - test: An object with a `calculate_t_test` method.
    - pt_exp: Experiment data handler.
    - num_list: A list of different numbers of traces to test.
    - target_byte: The target byte of the AES S-box.
    - grid_size: The grid size (default 5x5).
    """

    all_values = []  # To store separate lists for each number of traces
    labels = []  # Corresponding labels for box plots

    for num in num_list:
        print(f"Processing num_traces = {num}...")

        # Run the SNR heatmap function and get the matrix
        CEMA_values_rotated = plot_SNR_heatmap_byte_bp(test, pt_exp, num, target_byte, grid_size)

        # Ensure CEMA_values_rotated is a list of lists and flatten it properly
        if isinstance(CEMA_values_rotated, list):
            all_values.append(list(chain.from_iterable(CEMA_values_rotated)))  # Append as a separate list for each number of traces
        else:
            print(f"Invalid data format for num_traces={num}: {CEMA_values_rotated}")
            continue

        labels.append(f"{num} traces")  # Label each dataset

    # Create the box plot with the flattened data
    if all_values:  # Only plot if there is valid data
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=all_values)

        # Adjust labels
        plt.xticks(ticks=range(len(num_list)), labels=labels)
        plt.xlabel("Number of Traces")
        plt.ylabel("SNR Values")
        plt.title("Box Plot of SNR Values for Different Trace Counts")

        plt.show()
    else:
        print("No valid data to plot.")
        
    return all_values#,flat_values
    

def plot_t_statistic_heatmap(test, grid_size=5):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    # Initialize the t_values array directly
    t_values = np.zeros((grid_size, grid_size))

    # Calculate t-statistics and populate the t_values array in a single loop
    for i in range(grid_size):
        for j in range(grid_size):
            t, t_max = test.calculate_t_test(f"fixed_{i}_{j}", f"random_{i}_{j}")
            t_values[i, j] = np.nanmax(np.abs(t))  # Store the maximum absolute t-statistic

    # Rotate the heatmap by transposing the CEMA_values array
    t_values_rotated = np.rot90(t_values, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(t_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title
    plt.title("Heatmap of t-statistics")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()
    return t_values_rotated

## helper functions  

In [ ]:
def scapegoat_cpa(experiment):
    maxcpa = [0] * 256
    cparefs = [0] * 16 #put your key byte guess correlations here
    bestguess = [0] * 16 #put your key byte guesses here
    traces = experiment.get_dataset("CW_Capture_Traces").read_all()
    keys = experiment.get_dataset("CW_Capture_Keys").read_all()
    plaintexts = experiment.get_dataset("CW_Capture_Plaintexts").read_all()

    for j in trange(0,16):
        for k in range(0, 256):
            leakage = leakage_model_hamming_weight(num_traces=len(plaintexts), plaintexts=plaintexts, subkey_guess=k, target_byte=j)
            correlation = pearson_correlation(leakage, traces)
            maxcpa[k] =  np.nanmax(np.abs(correlation))

        bestguess[j] = np.argmax(maxcpa)
        cparefs[j] = np.nanmax(maxcpa)

    return bestguess,cparefs

def scapegoat_cpa_byte(traces,keys,plaintexts,target_byte):
    maxcpa = [0] * 256
    cparefs = [0] #put your key byte guess correlations here
    bestguess = [0]  #put your key byte guesses here
    for k in range(0, 256):
        leakage = leakage_model_hamming_weight(num_traces=len(plaintexts), plaintexts=plaintexts, subkey_guess=k, target_byte=target_byte)
        correlation = pearson_correlation(leakage, traces)
        maxcpa[k] =  np.nanmax(np.abs(correlation))

    bestguess = np.argmax(maxcpa)
    cparefs = np.nanmax(maxcpa)
    
    return bestguess,cparefs
def leakage_model_hamming_weight_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = bin(Sbox[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]).count('1')

    return leakage

def sbox_snr(num_traces: int, plaintexts: list | np.ndarray, subkey_guess: any, target_byte: int) -> np.ndarray:
    """
    Generates hypothetical leakage using the damming distance leakage model. In this implementation the reference state
    is the output of the sbox at index 0.

    :param num_traces: The number of traces collected when measuring the observed leakage
    :type num_traces: int
    :param plaintexts: The array of plaintexts used to collect the observed leakage
    :type plaintexts: list | np.ndarray
    :param subkey_guess: the subkey guess
    :type subkey_guess: any
    :param target_byte: the target byte of the key
    :type target_byte: int
    :return: numpy array of the hypothetical leakage
    :rtype: np.ndarray
    :Authors: Samuel Karkache (swkarkache@wpi.edu)
    """
    leakage = np.empty(num_traces, dtype=object)

    for i in range(num_traces):
        leakage[i] = Sbox[subkey_guess[i][target_byte] ^ plaintexts[i][target_byte]]

    return leakage

def plot_heatmap(heatmap_values, text, anno=False, cba=True, squar=True):
    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(heatmap_values, annot=anno, cbar=cba, square=squar)

    # Adding labels and title
    plt.title(text)
    plt.xlabel("X")
    plt.ylabel("Y")

    # Show the plot
    plt.show()
    return None
    
def save_mat(matr,file_name):
    # Example matrix
    matrix = np.random.rand(100, 100)  # Replace this with your actual matrix

    # Save to a .mat file
    scipy.io.savemat(file_name, {'matrix': matr})    
    return None

def save_fig(filename):
    plt.gcf().savefig("heatmap.svg", format="svg", bbox_inches="tight")



def plot_SNR_heatmap_byte_bp(test,pt_exp,num,target_byte =0, grid_size=5,SNR_type = "BYTE"):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    keys = pt_exp.get_dataset("keys").read_data(0,num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0,num)
    # Initialize the t_values array directly
    CEMA_values = [];#np.zeros((grid_size, grid_size))
#     CEMA_guesses = np.zeros((grid_size, grid_size))
    if SNR_type == "BYTE" :
        labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "HW":
        labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    elif SNR_type == "FULL"    :
        labels = Sbox[plaintexts ^ keys]
        labels = labels[:, 1] 
    else :
        print("incorrect SNR type")
        return -1
        

    labelsUnique = np.unique(labels)


    # Calculate t-statistics and populate the t_values array in a single loop
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {}
            for k in labelsUnique:
                sorted_labels[k] = []
            print(f"loop_{i}_{j}")

            traces = test.get_dataset( f"random_{i}_{j}").read_data(0,num)
            # organize labels using traces, keys and plaintexts


            # add traces to labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # run metric and visualize result. Save it in the visualization folder. 
            t = signal_to_noise_ratio(sorted_labels )#, visualize=True, visualization_path=test.get_visualization_path() + f"SNR_{i}_{j}")
            
            CEMA_values.append(t)   # Store the maximum absolute t-statistic

    return CEMA_values#,  10 * np.log10(CEMA_values)


# deprecated functions


In [ ]:
      
        
# def Grid_Tracing(X_range,Y_range,X_number_of_step,Y_number_of_step,X,Y,Z,interface,sco,number_of_traces):
#     cordinate_traces={}
#     X_moment=0
#     Y_moment=0
#     traces = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#     cordinate = (X_moment, Y_moment)
#     cordinate_traces[cordinate] = traces
#     X_start_position=X.get_actual_position()
#     Y_start_position=Y.get_actual_position()
#     print(f"Starting Postion - ({X_start_position},{Y_start_position})")
#     while Y_moment<=Y_number_of_step:
#         X_initial_position=X.get_actual_position()
#         Y_initial_position=Y.get_actual_position()

#         for _ in range(X_number_of_step):
#             X.move_by(X_range)
#             print(f'Moving X to {X_initial_position + X_range}')

#             while X.get_actual_position() != X_initial_position + X_range:


#                 time.sleep(0.1)
#             traces = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#             X_moment += 1
#             cordinate = (X_moment, Y_moment)
#             cordinate_traces[cordinate] = traces
#             X_initial_position=X.get_actual_position()

#         if Y_moment==Y_number_of_step:
#             break
#         Y.move_by(Y_range)
#         Y_moment+=1
#         print(f'Moving Y to {Y_initial_position + Y_range}')
#         while Y.get_actual_position() != Y_initial_position + Y_range:

#             time.sleep(0.1)
#         cordinate = (X_moment, Y_moment)
#         cordinate_traces[cordinate] = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#         Y_initial_position=Y.get_actual_position()

#         for _ in range(X_number_of_step):
#             X.move_by(-X_range)
#             print(f'Moving X to {X_initial_position - X_range}')
#             while X.get_actual_position() != X_initial_position - X_range:

#                 time.sleep(0.1)
#             traces = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#             X_moment -= 1
#             cordinate = (X_moment, Y_moment)
#             cordinate_traces[cordinate] = traces
#             X_initial_position=X.get_actual_position()
#         if Y_moment==Y_number_of_step:
#             break
#         # Move down 1 step
#         Y.move_by(Y_range)
#         Y_moment += 1
#         print(f'Moving Y to {Y_initial_position + Y_range}')
#         while Y.get_actual_position() != Y_initial_position + Y_range:
#             time.sleep(0.1)

#         cordinate = (X_moment, Y_moment)
#         cordinate_traces[cordinate] = [capture_nopt(sco,num_of_samples=500) for i in trange(number_of_traces)]
#         Y_initial_position=Y.get_actual_position()
    
#     X.move_to(X_start_position)
#     while X.get_actual_position() != X_start_position:
#         time.sleep(0.1)
#     Y.move_to(Y_start_position)
#     while Y.get_actual_position() != Y_start_position:
#         time.sleep(0.1)
#     print(f"Final Postion - ({X.get_actual_position()},{Y.get_actual_position()})")

#     return cordinate_traces


def plot_SNR_heatmap_byte(test,pt_exp,num,target_byte =0, grid_size=5):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    keys = pt_exp.get_dataset("keys").read_data(0,num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0,num)
    # Initialize the t_values array directly
    CEMA_values = np.zeros((grid_size, grid_size))
    labels = sbox_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    labelsUnique = np.unique(labels)


    # Calculate t-statistics and populate the t_values array in a single loop
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {}
            for k in labelsUnique:
                sorted_labels[k] = []
            print(f"loop_{i}_{j}")

            traces = test.get_dataset( f"random_{i}_{j}").read_data(0,num)
            # organize labels using traces, keys and plaintexts
            # add traces to labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # run metric and visualize result. Save it in the visualization folder. 
            t = signal_to_noise_ratio(sorted_labels )#, visualize=True, visualization_path=test.get_visualization_path() + f"SNR_{i}_{j}")
            
            CEMA_values[i, j] =  np.nanmax(np.abs(t))  # Store the maximum absolute t-statistic

    # Rotate the heatmap by transposing the CEMA_values array
    CEMA_values_rotated = np.rot90(CEMA_values, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title
    plt.title("Heatmap of SNR byte")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()
    return CEMA_values_rotated ,  10 * np.log10(CEMA_values_rotated)

def plot_SNR_heatmap_hw_byte(test,pt_exp,num,target_byte =0, grid_size=5):
    """
    Calculate t-statistics for a grid and plot the results as a heatmap.

    Parameters:
    - test: An object that has a `calculate_t_test` method to compute the t-statistics.
    - grid_size: The size of the grid (default is 5x5).
    """
    keys = pt_exp.get_dataset("keys").read_data(0,num)
    plaintexts = pt_exp.get_dataset("plaintexts").read_data(0,num)
    # Initialize the t_values array directly
    CEMA_values = np.zeros((grid_size, grid_size))
    CEMA_guesses = np.zeros((grid_size, grid_size))
    labels = leakage_model_hamming_weight_snr(num_traces=num, plaintexts=plaintexts, subkey_guess=keys, target_byte=target_byte)
    labelsUnique = np.unique(labels)
    # initialize the dictionary
    for i in range(grid_size):
        for j in trange(grid_size):
            sorted_labels = {}
            for k in labelsUnique:
                sorted_labels[k] = []
            print(f"loop_{i}_{j}")

            traces = test.get_dataset( f"random_{i}_{j}").read_data(0,num)
            # organize labels using traces, keys and plaintexts


            # add traces to labels
            for index, label in enumerate(labels):
                sorted_labels[label].append(np.array(traces[index]))

            # run metric and visualize result. Save it in the visualization folder. 
            t = signal_to_noise_ratio(sorted_labels);# , visualize=True, visualization_path=test.get_visualization_path() + f"SNR_{i}_{j}")
            
            CEMA_values[i, j] =  np.nanmax(np.abs(t))   # Store the maximum absolute t-statistic

    # Rotate the heatmap by transposing the CEMA_values array
    CEMA_values_rotated = np.rot90(CEMA_values, k=3)  # Rotate by 90 degrees clockwise (k=3)

    # Create a heatmap using seaborn
    plt.figure(figsize=(8, 6))
    sns.heatmap(CEMA_values_rotated, annot=True, cbar=True, square=True)

    # Adding labels and title
    plt.title("Heatmap of SNR byte HW")
    plt.xlabel("j")
    plt.ylabel("i")

    # Show the plot
    plt.show()
    return CEMA_values_rotated ,  10 * np.log10(CEMA_values_rotated)


